In [27]:
import pandas as pd
import glob
import os

# ============================================================
# 1. Автоматический поиск файла
# ============================================================
file_name = "sap500.csv"
search_paths = [
    file_name,                                  # В текущей папке
    f"sp500_volatility_project/data/raw/{file_name}", # По вашему изначальному пути
    f"data/raw/{file_name}",                    # Стандартный путь DVC/Cookiecutter
    f"../data/raw/{file_name}"                  # Если ноутбук лежит в папке notebooks/
]

file_path = None
for path in search_paths:
    if os.path.exists(path):
        file_path = path
        print(f"✅ Файл успешно найден: {file_path}")
        break

# Если не нашли в известных путях, ищем глобально
if not file_path:
    found = glob.glob(f'**/{file_name}', recursive=True)
    if found:
        file_path = found[0]
        print(f"✅ Файл найден через glob: {file_path}")
    else:
        raise FileNotFoundError(f"❌ Файл {file_name} не найден в проекте. Текущая CWD: {os.getcwd()}")

# ============================================================
# 2. Чтение исходного датасета
# ============================================================
df = pd.read_csv(file_path)

# Первичный осмотр структуры ДО преобразований:
# фиксируем форму, имена и типы столбцов, как они пришли из CSV.
print("Форма датасета:", df.shape)
print("Типы данных до преобразования:")
print(df.dtypes)
print("Первые строки до преобразования:")
print(df.head())

# ============================================================
# 3. Приведение столбца Date к типу datetime64[ns]
# ============================================================
# errors='coerce': любые некорректные значения дат будут превращены в NaT,
# чтобы их можно было явно обнаружить и обработать, а не получить скрытую ошибку.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# QA-проверка: сколько дат не удалось распознать.
# Ожидается 0, так как в METADATA_sap500.md формат YYYY-MM-DD и 0 пропусков.
print("Количество нераспознанных дат (NaT):", df["Date"].isna().sum())

# ============================================================
# 4. Строгая сортировка по Date по возрастанию
# ============================================================
# Критично для временных рядов: нарушение хронологии приводит к
# data leakage и некорректному расчёту лагов и доходностей.
# Фрагменты исходного файла показывают, что строки могут идти
# не в глобальном хронологическом порядке (блоки разных лет чередуются),
# поэтому сортировка обязательна даже если .head() выглядит корректно.
df = df.sort_values("Date", ascending=True).reset_index(drop=True)

# ============================================================
# 5. Проверки целостности временного ряда после сортировки
# ============================================================
print("Типы данных после преобразования:")
print(df.dtypes)

# Диапазон наблюдаемых дат: ожидаем старт в районе 1927-12-30
# (по фрагментам CSV) и окончание в районе 2026-05-11.
print("Минимальная дата:", df["Date"].min())
print("Максимальная дата:", df["Date"].max())

# Уникальность дат: обязательное условие, так как Date — первичный ключ (кандидат).
print("Date уникален:", df["Date"].is_unique)

# Монотонность по возрастанию после сортировки: должна быть True.
print("Date монотонно возрастает:", df["Date"].is_monotonic_increasing)

# Проверка на полные дубликаты строк и дубликаты по дате.
print("Полные дубликаты строк:", df.duplicated().sum())
print("Дубликаты по Date:", df["Date"].duplicated().sum())

# ============================================================
# 6. Контрольный срез данных после сортировки
# ============================================================
print("Первые строки после сортировки:")
print(df.head())
print("Последние строки после сортировки:")
print(df.tail())

✅ Файл успешно найден: ../data/raw/sap500.csv
Форма датасета: (24809, 6)
Типы данных до преобразования:
Date       object
Open      float64
High      float64
Low       float64
Close     float64
Volume    float64
dtype: object
Первые строки до преобразования:
         Date       Open       High        Low      Close  Volume
0  1927-12-30  17.660000  17.660000  17.660000  17.660000     0.0
1  1928-01-03  17.760000  17.760000  17.760000  17.760000     0.0
2  1928-01-04  17.719999  17.719999  17.719999  17.719999     0.0
3  1928-01-05  17.549999  17.549999  17.549999  17.549999     0.0
4  1928-01-06  17.660000  17.660000  17.660000  17.660000     0.0
Количество нераспознанных дат (NaT): 0
Типы данных после преобразования:
Date      datetime64[ns]
Open             float64
High             float64
Low              float64
Close            float64
Volume           float64
dtype: object
Минимальная дата: 1927-12-30 00:00:00
Максимальная дата: 2026-10-07 00:00:00
Date уникален: True
Date моното

---

## Промежуточный вывод по Этапу 1.1

### Что известно заранее (из `METADATA_sap500.md`)
- Датасет содержит **24 809 строк и 6 столбцов**.
- Столбцы: `Date`, `Open`, `High`, `Low`, `Close`, `Volume`.
- Пропусков нет: все 6 столбцов имеют ровно 24 809 non-null значений.
- `Date` в сыром файле имеет тип `object` и требует приведения к `datetime64[ns]`.
- В `.head()` даты идут по возрастанию, но глобальная монотонность не гарантирована — это и проверяется на данном этапе.

### Что нужно получить из notebook после выполнения кода
Чтобы подтвердить успешное прохождение Этапа 1.1, из вывода кода нужно явно зафиксировать следующие значения:

1. **`df.shape`** — ожидается `(24809, 6)`.
2. **`df.dtypes` до преобразования** — убедиться, что `Date` действительно имеет тип `object`, а остальные поля — `float64`.
3. **Количество NaT после `pd.to_datetime(..., errors='coerce')`** — ожидается `0`. Если значение ненулевое, это означает наличие нераспознанных дат, и такие строки нужно будет разобрать отдельно.
4. **`df['Date'].min()` и `df['Date'].max()`** — фактический диапазон наблюдений. По фрагментам данных ожидается старт около `1927-12-30` и окончание около `2026-05-11`.
5. **`df['Date'].is_unique`** — ожидается `True`. Если `False`, это означает наличие дублирующихся дат, что потребует обработки на следующем подэтапе (1.2 «Дубликаты»).
6. **`df['Date'].is_monotonic_increasing`** — после сортировки ожидается `True`.
7. **`df.duplicated().sum()` и `df['Date'].duplicated().sum()`** — ожидается `0` и `0`.
8. **`df.head()` и `df.tail()` после сортировки** — убедиться, что первые строки соответствуют самым ранним датам, последние — самым поздним, а цены выглядят согласованно (например, ранние значения близки к ~17, поздние — к тысячам пунктов).

### Логический итог этапа
- Данные считаны из `sap500.csv` без модификации исходного файла.
- Поле `Date` приведено к типу `datetime64[ns]`, что необходимо для корректной работы с временными рядами, лагами и скользящими окнами.
- Выполнена строгая сортировка по дате по возрастанию, индекс сброшен — это обязательное условие для предотвращения data leakage при последующем расчёте `Log_Return` и GARCH-моделировании.
- Проведены первичные проверки целостности ряда: уникальность дат, монотонность, отсутствие дубликатов.